# 19r — what explains the WIS? A mixed-model regression on the scenario and contact-model features

17j and 18j rank the 22 scenario series by mean WIS. A ranking does not say which ingredient buys the skill.
This notebook regresses the WIS of every forecast unit (series × origin × age bin, one fit per horizon) on
those ingredients, with mgcv `bam`.

**Response.** Log-scale WIS: the WIS after a log transform of forecast and observation, as in 9j/17j's
headline. Every value is strictly positive.

**Families**, all with a log link:
- **lognormal**: a gaussian fitted to `log(wis)`. Effects act on the **median** WIS.
- **gamma**: effects act on the **mean** WIS, which is what 17j/18j rank by.
- **Tweedie** (`tw()`): the power p ∈ (1, 2) is estimated, and effects act on the mean. Tweedie's distinctive
  feature is a point mass at 0, and WIS has no zeros, so p is expected to run to the upper end of mgcv's range
  (1.99). At that point it *is* the gamma.

**Predictors.** The reference is *no contact – no variant*, and `unweighted-negbin|mean` among the contact models.
- X_data:
  - `ctc_future`: contact, conditional on the t₀+h contacts;
  - `ctc_frozen`: contact frozen at t₀;
  - `variant`: the variant-specific renewal;
  - `ctc_future_variant`, `ctc_frozen_variant`: the interactions.
- X_model (0 for the two no-contact series):
  - `duration`: duration-weighted hurdle-Weibull degree;
  - `nbhd`: neighbourhood NGM;
  - `duration_nbhd`: their interaction;
  - `diagonal`: the mean-diagonal (no-interaction) NGM.
- The interactions are product columns rather than `a:b` terms, so each fixed term and its random slope share
  one name.

**Random effects.**
- Intercepts for origin, age group and origin × age. All 22 series are scored against one truth per
  (origin, age bin), and the interaction carries what they share. That is also why there is no incidence
  covariate: it is constant within that cell.
- **A random slope by origin on all 9 terms**, `s(origin, by = term, bs = "re")`. Each effect then has an average
  `b` plus a per-origin deviation `u_o ~ N(0, σ²)`.
  - Without the slopes, the 154 rows inside each origin count as independent evidence about `b`, and its interval
    comes out too narrow.
  - With them, `b`'s interval reflects how much the effect really varies across the 55 origins (§5 measures the
    widening).
  - `b + u_o` is the effect at that origin, drawn over time in §6.
  - `bam`'s slopes are independent of one another and of the intercept; no correlations are estimated.

**Estimation.** `bam(method = "fREML", discrete = TRUE, nthreads = detectCores() − 1)`. Every covariate is 0/1 or
a factor, so `discrete` loses nothing.
- Intervals are drawn from the Bayesian posterior approximation `N(b̂, Vp)`. That is conditional on the estimated
  variance components, so it is not fully Bayesian.
- Families are compared by AIC on the scale of `wis` itself: the lognormal fit's log-likelihood is moved from
  `log(wis)` to `wis` by its Jacobian, `−Σ log wis`.

**Window.** Origins ≤ 2021-11-01 (n = 55), as in 17j/18j's ranking. Each fit has 22 × 55 × 7 = 8 470 rows, and
the 12 fits take seconds each.

**Inputs.** Run 17j, then `julia --project=.. 19r_export_wis_age.jl` from `src/`. That script writes
`dt_intermediate/19r_wis_age_time_<token>.csv` and checks that the per-age WIS averages to 17j's per-origin WIS.
This notebook reads only that CSV. Outputs: `res/19r_*`. R deps: `mgcv`, `RhpcBLASctl` (`19r_utils.R`).

In [ ]:
source("19r_utils.R")
suppressPackageStartupMessages({ library(ggplot2); library(patchwork) })
options(width = 160, repr.plot.width = 16, repr.plot.height = 9)

TOKEN <- "temporal-w8h-lc0-eng-nuts"             # = contacts_label(cfg) in 17j/18j
PRE_OMICRON_END <- as.Date("2021-11-01")          # = 17j cell 24 / 18j
HORIZONS <- 1:4
NTHREADS <- parallel::detectCores() - 1
N_DRAWS <- 4000; SEED <- 1236
SAVE_DIR <- "../dt_intermediate"; RES <- "../res"; DPI <- 300
FAM_COLS <- scale_colour_manual(values = c(lognormal = "#1b9e77", gamma = "#d95f02", tweedie = "#7570b3"))
key_of <- function(k) { s <- strsplit(k, " ")[[1]]; list(family = s[1], horizon = as.integer(s[2])) }
cat(R.version.string, "| mgcv", format(packageVersion("mgcv")), "| nthreads", NTHREADS, "\n")

## 1. Data and design

In [ ]:
d_all <- fread(file.path(SAVE_DIR, sprintf("19r_wis_age_time_%s.csv", TOKEN)))
d_all[, forecast_date := as.Date(forecast_date)]
stopifnot(nrow(d_all) == 22 * 55 * length(HORIZONS) * 7, uniqueN(d_all$series) == 22,
          uniqueN(d_all$forecast_date) == 55, max(d_all$forecast_date) <= PRE_OMICRON_END,
          uniqueN(d_all$age_group) == 7, setequal(unique(d_all$horizon), HORIZONS), all(d_all$wis > 0),
          uniqueN(d_all[, .(series, forecast_date, horizon, age_group)]) == nrow(d_all))
d_all <- add_design(d_all)
AGE_LEVELS <- unique(d_all$age_group)[order(as.numeric(sub("[-+].*$", "", unique(d_all$age_group))))]
d_all[, `:=`(origin = factor(format(forecast_date)), age_group = factor(age_group, levels = AGE_LEVELS))]
d_all[, origin_age := interaction(origin, age_group, drop = TRUE)]
stopifnot(nlevels(d_all$origin_age) == 55 * 7)

design <- unique(d_all[, c("series", "scenario", X_TERMS), with = FALSE])
stopifnot(nrow(design) == 22, uniqueN(design[, ..X_TERMS]) == 22)   # one design row per series, all distinct
print(design[order(-ctc_future - ctc_frozen, scenario, series)], nrows = 30)
cat("design rank:", qr(cbind(1, as.matrix(design[, ..X_TERMS])))$rank, "of", length(X_TERMS) + 1, "\n")
stopifnot(qr(cbind(1, as.matrix(design[, ..X_TERMS])))$rank == length(X_TERMS) + 1)

# the raw response, for orientation
print(dcast(d_all[, .(mean_wis = round(mean(wis), 3)), by = .(scenario, horizon)], scenario ~ horizon,
            value.var = "mean_wis"))

## 2. The Tweedie parameterisation

The predictive check (§8) simulates Tweedie replicates with `rtweedie` (`19r_utils.R`). That simulator has to use
the same (μ, φ, p) as `tw()`.
1. mgcv's `ldTweedie` must equal the plain series Σₙ Pois(n; λ)·Gamma(y; n·a, s) on a grid. If it does, the two
   share one parameterisation.
2. The simulator's mean and variance must be μ and φμᵖ.

In [ ]:
tw_grid <- CJ(mu = c(0.05, 0.3, 1, 4), disp = c(0.05, 0.3, 1), pw = c(1.1, 1.5, 1.9), y = c(0.04, 0.2, 0.7, 2, 7))
tw_grid[, mgcv  := mapply(function(y, mu, p, phi) ldTweedie(y, mu, p = p, phi = phi)[1, 1], y, mu, pw, disp)]
tw_grid[, brute := mapply(dtweedie_log_brute, y, mu, disp, pw)]
err_ld <- max(abs(tw_grid$mgcv - tw_grid$brute))
set.seed(SEED); x <- rtweedie(2e6, 1.3, 0.4, 1.6)
err_sim <- c(mean = abs(mean(x) / 1.3 - 1), var = abs(var(x) / (0.4 * 1.3^1.6) - 1))
cat(sprintf("ldTweedie vs series: %.1e over %d points | simulator mean, var rel err: %.1e, %.1e\n",
            err_ld, nrow(tw_grid), err_sim["mean"], err_sim["var"]))
stopifnot(err_ld < 1e-9, all(err_sim < 0.01))

## 3. Fits

Three families × four horizons. Every fit has 8 470 rows, 10 fixed coefficients and 12 random-effect terms.

In [ ]:
fits <- list()
for (h in HORIZONS) for (fam in FAMILIES) {
  t0 <- Sys.time()
  fits[[paste(fam, h)]] <- fit_wis_bam(d_all[horizon == h], fam, nthreads = NTHREADS)
  cat(sprintf("%-9s h%d: %5.1f s\n", fam, h, as.numeric(difftime(Sys.time(), t0, units = "secs"))))
}

## 4. Fit summary and family comparison

- **Convergence**: `converged` is `bam`'s outer-iteration flag.
- **`sig2`** is the scale: σ² for the lognormal and φ for gamma and Tweedie.
- **Tweedie p**: a value at 1.99 (mgcv's default upper limit for `tw()`) means the family has collapsed onto the
  gamma. That is a result about the data (no zeros), not a failure.
- **Jacobian check**: the lognormal's log-likelihood for `log(wis)`, minus Σ log wis, must equal Σ `dlnorm` at the
  ML scale. That identity is what licenses comparing its AIC with the other two.
- **AIC**: AIC on the `wis` scale, with ΔAIC from the best family at each horizon.

In [ ]:
fit_sum <- rbindlist(lapply(names(fits), function(k) {
  f <- fits[[k]]; kk <- key_of(k); y <- d_all[horizon == kk$horizon, wis]
  data.table(family = kk$family, horizon = kk$horizon, converged = isTRUE(f$converged),
             dev_expl = summary(f)$dev.expl, sig2 = f$sig2, tweedie_p = tweedie_p(f),
             edf = sum(f$edf), aic_wis = aic_on_wis(f, kk$family, y))
}))
fit_sum[, tweedie_at_bound := !is.na(tweedie_p) & tweedie_p >= 1.98]
fit_sum[, delta_aic := aic_wis - min(aic_wis), by = horizon]
setorder(fit_sum, horizon, delta_aic)
print(fit_sum)
stopifnot(all(fit_sum$converged))
if (any(fit_sum$tweedie_at_bound)) message("Tweedie p at mgcv's upper limit (h", paste(fit_sum[(tweedie_at_bound), horizon], collapse = ", "),
                                           "): the Tweedie has collapsed onto the gamma there.")
fwrite(fit_sum, file.path(RES, "19r_fit_summary.csv"))
fwrite(fit_sum[, .(horizon, family, aic_wis, delta_aic)], file.path(RES, "19r_aic_compare.csv"))

# The Jacobian check, at every horizon.
for (h in HORIZONS) {
  f <- fits[[paste("lognormal", h)]]; y <- d_all[horizon == h, wis]
  stopifnot(isTRUE(all.equal(log(y), as.numeric(f$y))))
  s_ml <- sqrt(sum((f$y - fitted(f))^2) / length(y))
  gap <- abs((as.numeric(logLik(f)) - sum(log(y))) - sum(dlnorm(y, fitted(f), s_ml, log = TRUE)))
  stopifnot(gap < 1e-6)
}
cat("Jacobian check passed at h", paste(HORIZONS, collapse = ", "), "\n")

BEST_FAM <- fit_sum[, .(aic = sum(aic_wis)), by = family][which.min(aic), family]
cat("lowest total AIC over h1–4:", BEST_FAM, "\n")

## 5. Effects on WIS

Each row is a **multiplicative effect on WIS**, `exp(b)`, with the median and 90% interval of draws from
`N(b̂, Vp)`. `pr_lt1` = P(effect < 1), the probability that the term *lowers* WIS, so < 1 is better.
- Lognormal's effects act on the median WIS, and gamma's and Tweedie's on the mean. The family is drawn in colour,
  so a split between families shows where that difference matters.
- The shaded rows are contrasts derived from the coefficients, draw by draw:
  - future vs frozen contacts, with and without the variant renewal (< 1 = future contacts beat frozen ones);
  - the variant effect with future contacts and with frozen contacts (with no contact it is `variant` itself);
  - each contact-model main effect evaluated at the other factor's non-reference level.

In [ ]:
draws <- lapply(fits, coef_draws, n = N_DRAWS, seed = SEED)
eff <- rbindlist(lapply(names(fits), function(k) { kk <- key_of(k); summarise_effects(fixed_effect_draws(draws[[k]]), kk$family, kk$horizon) }))
setcolorder(eff, c("family", "horizon", "term"))
fwrite(eff, file.path(RES, "19r_fixed_effects.csv"))
print(dcast(eff[, .(term, family, horizon, ratio = round(ratio_median, 3))], term ~ family + horizon, value.var = "ratio"), nrows = 30)

TERM_LEVELS <- rev(c(X_TERMS, DERIVED_TERMS))
p_eff <- ggplot(eff[, term := factor(term, levels = TERM_LEVELS)],
                aes(x = ratio_median, xmin = ratio_lo, xmax = ratio_hi, y = term, colour = family)) +
  annotate("rect", xmin = min(eff$ratio_lo), xmax = Inf, ymin = 0.5, ymax = length(DERIVED_TERMS) + 0.5, fill = "grey92", colour = NA) +
  geom_vline(xintercept = 1, linetype = 2, colour = "grey40") +
  geom_pointrange(position = position_dodge(width = 0.6), size = 0.25) +
  scale_x_log10() + facet_grid(. ~ horizon, labeller = label_both) + FAM_COLS +
  labs(x = "multiplicative effect on log-scale WIS (median, 90% interval; < 1 = better)", y = NULL,
       title = sprintf("19r — effects on WIS by horizon, origins ≤ %s (n = 55); shaded = derived contrasts", PRE_OMICRON_END)) +
  theme_bw(base_size = 11) + theme(legend.position = "bottom")
ggsave(file.path(RES, "19r_fixed_effects.png"), p_eff, width = 16, height = 8, dpi = DPI)
p_eff

### Tweedie only: one forest plot across all terms

The same effects for the Tweedie fit alone, on one panel. The four horizons are dodged colours within each term,
so a term's change with lead time reads along its row. The rows are grouped as X_data, X_model, then the derived
contrasts (shaded), with a dashed line between the first two groups.

In [ ]:
eff_tw <- eff[family == "tweedie"][, `:=`(term = factor(as.character(term), levels = TERM_LEVELS), horizon = factor(horizon))]
stopifnot(nrow(eff_tw) == length(TERM_LEVELS) * length(HORIZONS))
N_XDATA <- 5   # ctc_future … ctc_frozen_variant come first in X_TERMS
p_tw <- ggplot(eff_tw, aes(x = ratio_median, xmin = ratio_lo, xmax = ratio_hi, y = term, colour = horizon)) +
  annotate("rect", xmin = min(eff_tw$ratio_lo), xmax = Inf, ymin = 0.5, ymax = length(DERIVED_TERMS) + 0.5,
           fill = "grey92", colour = NA) +
  geom_hline(yintercept = length(TERM_LEVELS) - N_XDATA + 0.5, colour = "grey60", linetype = 3) +
  geom_vline(xintercept = 1, linetype = 2, colour = "grey40") +
  geom_pointrange(position = position_dodge(width = 0.7, reverse = TRUE), size = 0.3) +
  scale_x_log10(breaks = c(0.7, 0.8, 0.9, 1, 1.1, 1.25, 1.5, 1.75)) +
  scale_colour_viridis_d(end = 0.85, name = "horizon (weeks)") +
  labs(x = "multiplicative effect on log-scale WIS (median, 90% interval; < 1 = better)", y = NULL,
       title = sprintf("19r — Tweedie: effects on WIS, origins ≤ %s (n = 55)", PRE_OMICRON_END),
       subtitle = "top: X_data · middle: X_model · shaded: derived contrasts") +
  theme_bw(base_size = 12) + theme(legend.position = "bottom")
ggsave(file.path(RES, "19r_fixed_effects_tweedie.png"), p_tw, width = 10, height = 9, dpi = DPI)
p_tw

### How much the random slopes widened the intervals

Below, `BEST_FAM` is refitted at h1 with the random intercepts only, and the standard error of each of the 9 terms
is compared. A ratio above 1 is the precision that ignoring the variation across origins would have falsely claimed.

In [ ]:
f_int <- fit_wis_bam(d_all[horizon == 1], BEST_FAM, slopes = FALSE, nthreads = NTHREADS)
f_slp <- fits[[paste(BEST_FAM, 1)]]
se_cmp <- data.table(term = X_TERMS, b_slopes = coef(f_slp)[X_TERMS], b_intercepts = coef(f_int)[X_TERMS],
                     se_slopes = sqrt(diag(vcov(f_slp)))[X_TERMS], se_intercepts = sqrt(diag(vcov(f_int)))[X_TERMS])
se_cmp[, se_ratio := se_slopes / se_intercepts]
print(se_cmp, digits = 3)
fwrite(se_cmp, file.path(RES, "19r_se_slopes_vs_intercepts_h1.csv"))

## 6. The effects over time: random slopes by origin

`exp(b + u_o)` is the effect of each term at each origin, with a 90% interval. It is drawn for `BEST_FAM` (lowest
total AIC), one panel per term and one colour per horizon, with 9j's periods shaded. The CSV holds every family.

A slope whose SD is estimated at ≈ 0 (§7) gives a flat line at the average effect: that term does not vary
measurably across origins.

In [ ]:
slopes <- rbindlist(lapply(names(fits), function(k) {
  kk <- key_of(k)
  rbindlist(lapply(X_TERMS, function(t) origin_slope_frame(draws[[k]], fits[[k]], t)))[, `:=`(family = kk$family, horizon = kk$horizon)]
}))
setcolorder(slopes, c("family", "horizon", "term", "origin"))
fwrite(slopes, file.path(RES, "19r_origin_slopes.csv"))

sl <- slopes[family == BEST_FAM][, `:=`(term = factor(term, levels = X_TERMS), horizon = factor(horizon))]
per <- PERIODS[start <= max(sl$origin) & end >= min(sl$origin)]
p_sl <- ggplot(sl) +
  geom_rect(data = per, aes(xmin = start, xmax = end, ymin = min(sl$ratio_lo), ymax = Inf, fill = period), alpha = 0.15, inherit.aes = FALSE) +
  geom_hline(yintercept = 1, linetype = 2, colour = "grey40") +
  geom_ribbon(aes(x = origin, ymin = ratio_lo, ymax = ratio_hi, group = horizon), fill = "grey50", alpha = 0.12) +
  geom_line(aes(x = origin, y = ratio_median, colour = horizon)) +
  scale_y_log10() + facet_wrap(~ term, ncol = 3) +
  scale_colour_viridis_d(end = 0.85, name = "horizon") + scale_fill_brewer(palette = "Pastel1", name = "period (9j)") +
  labs(x = "forecast origin", y = "exp(b + u_origin), 90% interval",
       title = sprintf("19r — effect of each term by origin (%s), < 1 = lowers WIS", BEST_FAM)) +
  theme_bw(base_size = 11) + theme(legend.position = "bottom")
ggsave(file.path(RES, "19r_origin_slopes.png"), p_sl, width = 16, height = 11, dpi = DPI)
p_sl

## 7. Random effects

SDs are on the log scale, with mgcv's 95% interval (`gam.vcomp`). An SD near 0 with an interval spanning many
orders of magnitude means that term is shrunk to nothing: there is no detectable variation across origins.
`at_zero` flags those.

The age and origin intercepts are drawn as `exp(r)`.

In [ ]:
re_sd <- rbindlist(lapply(names(fits), function(k) cbind(as.data.table(key_of(k)), re_sd_table(fits[[k]]))))
re_sd[, at_zero := sd < 0.01 | sd_hi / sd_lo > 1e6]
fwrite(re_sd, file.path(RES, "19r_re_sd.csv"))
print(dcast(re_sd[, .(term, family, horizon, sd = round(sd, 3))], term ~ family + horizon, value.var = "sd"), nrows = 20)
cat("terms at zero (family h):\n"); print(re_sd[(at_zero), .(fits = paste(family, horizon, collapse = ", ")), by = term])

re_age <- rbindlist(lapply(names(fits), function(k) cbind(as.data.table(key_of(k)), re_level_frame(draws[[k]], fits[[k]], "age_group"))))
re_age[, level := factor(level, levels = AGE_LEVELS)]
p_age <- ggplot(re_age, aes(x = level, y = ratio_median, ymin = ratio_lo, ymax = ratio_hi, colour = family)) +
  geom_hline(yintercept = 1, linetype = 2, colour = "grey40") +
  geom_pointrange(position = position_dodge(width = 0.5), size = 0.25) + scale_y_log10() +
  facet_grid(. ~ horizon, labeller = label_both) + FAM_COLS +
  labs(x = "age group", y = "exp(random intercept), 90% interval", title = "19r — age-group random intercepts") +
  theme_bw(base_size = 11) + theme(legend.position = "bottom", axis.text.x = element_text(angle = 45, hjust = 1))
ggsave(file.path(RES, "19r_age_re.png"), p_age, width = 16, height = 5, dpi = DPI)
p_age

re_origin <- rbindlist(lapply(names(fits), function(k) cbind(as.data.table(key_of(k)), re_level_frame(draws[[k]], fits[[k]], "origin"))))
re_origin[, origin := as.Date(level)]
p_origin <- ggplot(re_origin) +
  geom_rect(data = per, aes(xmin = start, xmax = end, ymin = min(re_origin$ratio_lo), ymax = Inf, fill = period), alpha = 0.15, inherit.aes = FALSE) +
  geom_hline(yintercept = 1, linetype = 2, colour = "grey40") +
  geom_ribbon(aes(x = origin, ymin = ratio_lo, ymax = ratio_hi, group = family), fill = "grey50", alpha = 0.15) +
  geom_line(aes(x = origin, y = ratio_median, colour = family)) +
  scale_y_log10() + facet_grid(horizon ~ ., labeller = label_both) + FAM_COLS +
  scale_fill_brewer(palette = "Pastel1", name = "period (9j)") +
  labs(x = "forecast origin", y = "exp(random intercept), 90% interval", title = "19r — origin random intercepts") +
  theme_bw(base_size = 11) + theme(legend.position = "bottom")
ggsave(file.path(RES, "19r_origin_re.png"), p_origin, width = 14, height = 10, dpi = DPI)
p_origin

## 8. Posterior predictive checks

The observed density of WIS (black) against 50 replicates per family (colour), at h1 and h4, on a log x-axis.
Replicates are simulated from the fitted means and the estimated scale, without parameter uncertainty
(`simulate_wis`). Tweedie replicates that are exactly 0 (its point mass) cannot be drawn on a log axis, and the
title counts them. The x-axis is the observed range, widened by a factor of 3 each side.

In [ ]:
pp_plots <- list()
for (h in c(1, 4)) for (fam in FAMILIES) {
  f <- fits[[paste(fam, h)]]; y <- d_all[horizon == h, wis]
  sims <- simulate_wis(f, fam, 50, seed = SEED + h)
  n0 <- sum(sims == 0)
  rep_dt <- data.table(rep = rep(seq_len(nrow(sims)), ncol(sims)), wis = as.vector(sims))[wis > 0]
  pp_plots[[paste(fam, h)]] <- ggplot() +
    geom_density(data = rep_dt, aes(x = wis, group = rep), colour = "#7fa7c9", linewidth = 0.2) +
    geom_density(data = data.table(wis = y), aes(x = wis), colour = "black", linewidth = 0.8) +
    scale_x_log10() + coord_cartesian(xlim = range(y) * c(1 / 3, 3)) +
    labs(title = sprintf("%s, h%d%s", fam, h, if (n0 > 0) sprintf(" (%d zeros not drawn)", n0) else ""),
                           x = "log-scale WIS", y = "density") + theme_bw(base_size = 10)
}
p_pp <- wrap_plots(pp_plots, nrow = 2) + plot_annotation(title = "19r — observed (black) vs simulated WIS (50 replicates)")
ggsave(file.path(RES, "19r_ppc.png"), p_pp, width = 15, height = 8, dpi = DPI)
p_pp

In [ ]:
# Everything this notebook wrote.
print(file.info(list.files(RES, pattern = "^19r_", full.names = TRUE))[, c("size", "mtime")])